# Feature Extraction: One-Hot, Bag of Words, N-Grams, TF-IDF

Converting a small toy corpus of text into numerical representations, comparing sparsity and the information each representation captures.

## Toy corpus

A small hardcoded corpus of sentences about two topics (cats and cars) so the differences between representations are easy to see.

In [1]:
corpus = [
    "the cat sat on the mat",
    "the cat is not happy today",
    "the dog is happy and playful",
    "a fast car is on the road",
    "the fast car is not on the mat",
    "cats and dogs are great pets",
    "the road is long and the car is fast",
    "my cat and my dog play together",
]
for i, doc in enumerate(corpus):
    print(f"{i}: {doc}")

0: the cat sat on the mat
1: the cat is not happy today
2: the dog is happy and playful
3: a fast car is on the road
4: the fast car is not on the mat
5: cats and dogs are great pets
6: the road is long and the car is fast
7: my cat and my dog play together


## One-Hot Encoding of Text

Build a one-hot vector per token using `pandas.get_dummies` on the flattened token list, then show how sparse and similarity-blind the representation is.

In [2]:
import pandas as pd

tokens = corpus[0].split()
print(f"Tokens in doc 0: {tokens}")

one_hot = pd.get_dummies(tokens)
one_hot.index = tokens
one_hot

Tokens in doc 0: ['the', 'cat', 'sat', 'on', 'the', 'mat']


,cat,mat,on,sat,the
the,False,False,False,False,True
cat,True,False,False,False,False
sat,False,False,False,True,False
on,False,False,True,False,False
the,False,False,False,False,True
mat,False,True,False,False,False


In [3]:
import numpy as np

# Any two distinct words are exactly as "different" as any other two under one-hot encoding
words_unique = list(dict.fromkeys(tokens))
oh = pd.get_dummies(words_unique)
oh.index = words_unique

def cosine_sim(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    return float(a @ b) / denom if denom else 0.0

print("cosine(cat, mat)  =", cosine_sim(oh.loc["cat"], oh.loc["mat"]))
print("cosine(cat, sat)  =", cosine_sim(oh.loc["cat"], oh.loc["sat"]))
print("Every pair of distinct one-hot vectors is orthogonal -> no notion of similarity, and the matrix is mostly zeros.")
oh

cosine(cat, mat)  = 0.0
cosine(cat, sat)  = 0.0
Every pair of distinct one-hot vectors is orthogonal -> no notion of similarity, and the matrix is mostly zeros.


,cat,mat,on,sat,the
the,False,False,False,False,True
cat,True,False,False,False,False
sat,False,False,False,True,False
on,False,False,True,False,False
mat,False,True,False,False,False


## Bag of Words (BOW)

Use `CountVectorizer` to build a unigram word-count matrix over the whole corpus.

In [4]:
from sklearn.feature_extraction.text import CountVectorizer

bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(corpus)

bow_df = pd.DataFrame(bow_matrix.toarray(), columns=bow_vectorizer.get_feature_names_out())
print(f"BOW matrix shape: {bow_df.shape}")
bow_df

BOW matrix shape: (8, 24)


,and,are,car,cat,cats,dog,dogs,fast,great,happy,...,not,on,pets,play,playful,road,sat,the,today,together
0,0,0,0,1,0,0,0,0,0,0,...,0,1,0,0,0,0,1,2,0,0
1,0,0,0,1,0,0,0,0,0,1,...,1,0,0,0,0,0,0,1,1,0
2,1,0,0,0,0,1,0,0,0,1,...,0,0,0,0,1,0,0,1,0,0
3,0,0,1,0,0,0,0,1,0,0,...,0,1,0,0,0,1,0,1,0,0
4,0,0,1,0,0,0,0,1,0,0,...,1,1,0,0,0,0,0,2,0,0
5,1,1,0,0,1,0,1,0,1,0,...,0,0,1,0,0,0,0,0,0,0
6,1,0,1,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,2,0,0
7,1,0,0,1,0,1,0,0,0,0,...,0,0,0,1,0,0,0,0,0,1


## N-Grams (Bigrams)

Use `CountVectorizer(ngram_range=(2, 2))` to extract only bigrams, which capture short local phrases (like negation: 'not happy') that unigram BOW cannot distinguish from independent 'not' and 'happy' counts.

In [5]:
bigram_vectorizer = CountVectorizer(ngram_range=(2, 2))
bigram_matrix = bigram_vectorizer.fit_transform(corpus)

bigram_df = pd.DataFrame(bigram_matrix.toarray(), columns=bigram_vectorizer.get_feature_names_out())
print(f"Bigram matrix shape: {bigram_df.shape}")
bigram_df

Bigram matrix shape: (8, 37)


,and dogs,and my,and playful,and the,are great,car is,cat and,cat is,cat sat,cats and,...,on the,play together,road is,sat on,the car,the cat,the dog,the fast,the mat,the road
0,0,0,0,0,0,0,0,0,1,0,...,1,0,0,1,0,1,0,0,1,0
1,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
2,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
3,0,0,0,0,0,1,0,0,0,0,...,1,0,0,0,0,0,0,0,0,1
4,0,0,0,0,0,1,0,0,0,0,...,1,0,0,0,0,0,0,1,1,0
5,1,0,0,0,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
6,0,0,0,1,0,1,0,0,0,0,...,0,0,1,0,1,0,0,0,0,1
7,0,1,0,0,0,0,1,0,0,0,...,0,1,0,0,0,0,0,0,0,0


In [6]:
# Unigram BOW cannot tell "not happy" (doc 1) from "happy" alone (doc 2) beyond co-occurrence counts of
# the two words independently. Bigrams capture the phrase directly:
print("Docs containing the bigram 'not happy':")
print(bigram_df[bigram_df["not happy"] > 0].index.tolist())
print(corpus[1])

Docs containing the bigram 'not happy':
[1]
the cat is not happy today


## TF-IDF

Use `TfidfVectorizer` to weight terms by how characteristic they are of a document relative to the corpus.

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out()).round(3)
print(f"TF-IDF matrix shape: {tfidf_df.shape}")
tfidf_df

TF-IDF matrix shape: (8, 24)


,and,are,car,cat,cats,dog,dogs,fast,great,happy,...,not,on,pets,play,playful,road,sat,the,today,together
0,0.000,0.00,0.000,0.374,0.00,0.000,0.00,0.000,0.00,0.000,...,0.000,0.374,0.00,0.000,0.000,0.000,0.517,0.516,0.000,0.000
1,0.000,0.00,0.000,0.387,0.00,0.000,0.00,0.000,0.00,0.448,...,0.448,0.000,0.00,0.000,0.000,0.000,0.000,0.267,0.535,0.000
2,0.345,0.00,0.000,0.000,0.00,0.456,0.00,0.000,0.00,0.456,...,0.000,0.000,0.00,0.000,0.545,0.000,0.000,0.272,0.000,0.000
3,0.000,0.00,0.429,0.000,0.00,0.000,0.00,0.429,0.00,0.000,...,0.000,0.429,0.00,0.000,0.000,0.498,0.000,0.297,0.000,0.000
4,0.000,0.00,0.349,0.000,0.00,0.000,0.00,0.349,0.00,0.000,...,0.405,0.349,0.00,0.000,0.000,0.000,0.000,0.483,0.000,0.000
5,0.273,0.43,0.000,0.000,0.43,0.000,0.43,0.000,0.43,0.000,...,0.000,0.000,0.43,0.000,0.000,0.000,0.000,0.000,0.000,0.000
6,0.273,0.00,0.311,0.000,0.00,0.000,0.00,0.311,0.00,0.000,...,0.000,0.000,0.00,0.000,0.000,0.360,0.000,0.430,0.000,0.000
7,0.230,0.00,0.000,0.262,0.00,0.303,0.00,0.000,0.00,0.000,...,0.000,0.000,0.00,0.362,0.000,0.000,0.000,0.000,0.000,0.362


In [8]:
# Compare: "the" is common across almost every document (low IDF, so low TF-IDF weight)
# while a rarer word like "playful" gets a much higher relative weight.
comparison = pd.DataFrame({
    "bow_the": bow_df["the"],
    "tfidf_the": tfidf_df["the"],
    "bow_playful": bow_df["playful"],
    "tfidf_playful": tfidf_df["playful"],
})
comparison

,bow_the,tfidf_the,bow_playful,tfidf_playful
0,2,0.516,0,0.000
1,1,0.267,0,0.000
2,1,0.272,1,0.545
3,1,0.297,0,0.000
4,2,0.483,0,0.000
5,0,0.000,0,0.000
6,2,0.430,0,0.000
7,0,0.000,0,0.000


### Summary

- **One-hot**: sparse, one dimension per token occurrence, zero notion of similarity between words.
- **BOW**: sparse count matrix over the vocabulary, retains frequency but discards word order.
- **Bigrams**: capture local word-order context (e.g. negation) that unigram BOW misses, at the cost of a larger, sparser vocabulary.
- **TF-IDF**: reweights BOW counts to downweight corpus-common words (like 'the') and upweight document-distinctive words (like 'playful').